<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/docs/labs/projeto1/projeto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Java setup (required for pyspark)

In [1]:
#@title Java Setup (needed for pyspark)
!apt-get install -y openjdk-17-jre 2>/dev/null > /dev/null

#Download 1% sample

In [2]:
#@title Download 1% sample
!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

#Inspect dataset schema

In [3]:
#@title Dataset Schema
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]').appName('taxis').getOrCreate()

try :
    data = spark.read.csv('taxi_rides_1pc.csv.gz', sep =',', header=True, inferSchema=True)

    data.printSchema()

except Exception as err:
    print(err)

root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



#Register latlon_to_grid() and inBounds() functions as user defined functions (UDF)

In [4]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, IntegerType, BooleanType, DoubleType

# Longitude and latitude from the upper left corner of the grid
MIN_LON = -74.916578
MAX_LAT = 41.47718278

# Longitude and latitude that correspond to a shift in 500 meters
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

def latlon_to_grid(lat, lon):
    return ((int)((MAX_LAT - lat)/LAT_DELTA), (int)((lon - MIN_LON)/LON_DELTA))

def inBounds( cell ):
    return cell[0] > 0 and cell[0] < 300 and cell[1] > 0 and cell[1] < 300

# Register latlon_to_grid as a UDF
latlon_to_grid_udf = udf(latlon_to_grid, ArrayType(IntegerType()))

# Register inBounds as a UDF
inBounds_udf = udf(inBounds, BooleanType())

#Clean the data by removing null and invalid coordinates

In [5]:
from pyspark.sql.functions import col, isnan, when

# Filter out rows with null or invalid coordinate values before applying UDFs
# Check for both None and NaN if the schema is double
cleaned_data = data.filter(
    col("pickup_latitude").isNotNull() & ~isnan(col("pickup_latitude")) &
    col("pickup_longitude").isNotNull() & ~isnan(col("pickup_longitude")) &
    col("dropoff_latitude").isNotNull() & ~isnan(col("dropoff_latitude")) &
    col("dropoff_longitude").isNotNull() & ~isnan(col("dropoff_longitude"))
)

#Convert the coordinates using the helper functions

In [6]:
# Apply latlon_to_grid_udf to pickup and dropoff coordinates
data_with_grid_coords = cleaned_data \
    .withColumn("pickup_grid_coords", latlon_to_grid_udf(col("pickup_latitude"), col("pickup_longitude"))) \
    .withColumn("dropoff_grid_coords", latlon_to_grid_udf(col("dropoff_latitude"), col("dropoff_longitude")))

# Extract x and y coordinates
data_with_grid_coords = data_with_grid_coords \
    .withColumn("pickup_grid_x", col("pickup_grid_coords").getItem(0)) \
    .withColumn("pickup_grid_y", col("pickup_grid_coords").getItem(1)) \
    .withColumn("dropoff_grid_x", col("dropoff_grid_coords").getItem(0)) \
    .withColumn("dropoff_grid_y", col("dropoff_grid_coords").getItem(1))

#Filter the data by removing entries that are outside the grid

In [7]:
# Filter DataFrame to include only trips within the 300x300 grid for both pickup and dropoff
filtered_data = data_with_grid_coords.filter(
    inBounds_udf(col("pickup_grid_coords")) & inBounds_udf(col("dropoff_grid_coords"))
)

#Calculate the trips profit

In [8]:
# Calculate trip_profit
final_data = filtered_data.withColumn("trip_profit", col("fare_amount") + col("tip_amount"))

#Display data reduction and final data schema

In [9]:
print("Original DataFrame count:", data.count())
print("Cleaned DataFrame count (after removing null coords):", cleaned_data.count())
print("DataFrame with grid coordinates, filtered and profit count:", final_data.count())
final_data.printSchema()

Original DataFrame count: 1735010
Cleaned DataFrame count (after removing null coords): 1734973
DataFrame with grid coordinates, filtered and profit count: 1699848
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_grid_coords: array (nullable = true)
 |    |-- el

#Determine profitability per pickup area

The profitability of a pickup area is determined by dividing the area profit by the number of empty taxis in that area within the last 15 minutes:

*   The area profit is computed by calculating the average profit (fare+tip) for trips that started in the area and ended within the last 15 minutes.
*   The number of empty taxis in the area is the sum of taxis that had a drop-off location in that area less than 30 minutes ago and had no following pickup yet.

To determine the average profit per pickup area, we group the filtered DataFrame by the pickup_grid_x and pickup_grid_y columns (representing the pickup grid cell) and compute the average trip_profit for each group.

##Calculate average profit per pickup area

In [10]:
from pyspark.sql.functions import avg

# Group by pickup grid coordinates and calculate the average trip_profit
avg_profit_per_pickup_area = final_data.groupBy("pickup_grid_x", "pickup_grid_y").agg(avg("trip_profit").alias("avg_trip_profit"))

# Display the schema and first few rows of the resulting DataFrame
print("Schema of avg_profit_per_pickup_area:")
avg_profit_per_pickup_area.printSchema()
print("\nFirst 10 rows of avg_profit_per_pickup_area:")
avg_profit_per_pickup_area.show(10)

Schema of avg_profit_per_pickup_area:
root
 |-- pickup_grid_x: integer (nullable = true)
 |-- pickup_grid_y: integer (nullable = true)
 |-- avg_trip_profit: double (nullable = true)


First 10 rows of avg_profit_per_pickup_area:
+-------------+-------------+------------------+
|pickup_grid_x|pickup_grid_y|   avg_trip_profit|
+-------------+-------------+------------------+
|          172|          151|18.723061180984892|
|          181|          173|              17.0|
|          180|          154|17.041356783919593|
|          177|          163| 22.70818181818182|
|          175|          178| 18.75166666666667|
|          161|          175|             9.375|
|          119|          137|              13.9|
|          138|          172|              11.0|
|          201|          114|              8.75|
|          159|          147|             9.075|
+-------------+-------------+------------------+
only showing top 10 rows



##Determine which taxis are empty
For each trip, identify the pickup_datetime of the next trip for the same taxi using Spark window functions. Calculate the time difference between the current trip's dropoff_datetime and the pickup_datetime of the subsequent trip. Flag a trip's drop-off location as contributing to the number of empty taxis if this time difference exceeds 30 minutes, or if it is the last recorded trip for that taxi.

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, datediff, unix_timestamp, when, lit

# Define the window specification: partition by medallion, order by pickup_datetime
window_spec = Window.partitionBy("medallion").orderBy("pickup_datetime")

# Add next_pickup_datetime
data_with_next_pickup = final_data.withColumn(
    "next_pickup_datetime",
    lead(col("pickup_datetime"), 1).over(window_spec)
)

# Calculate time_to_next_pickup_minutes
# Convert timestamps to Unix timestamps (seconds) for accurate difference calculation
data_with_next_pickup = data_with_next_pickup.withColumn(
    "time_to_next_pickup_minutes",
    (unix_timestamp(col("next_pickup_datetime")) - unix_timestamp(col("dropoff_datetime"))) / 60
)

# Create is_empty_taxi status
# Flag data for which the next_pickup_datetime is null (last trip) or the time_to_next_pickup_minutes is > 30
final_data_with_empty_status = data_with_next_pickup.withColumn(
    "is_empty_taxi",
    when(
        (col("next_pickup_datetime").isNull()) | (col("time_to_next_pickup_minutes") > 30),
        True
    ).otherwise(False)
)

# Display schema and first few rows
print("Schema of DataFrame with empty taxi status:")
final_data_with_empty_status.printSchema()
print("\nFirst 10 rows of DataFrame with empty taxi status (showing relevant columns):")
final_data_with_empty_status.select(
    "medallion", "pickup_datetime", "dropoff_datetime",
    "next_pickup_datetime", "time_to_next_pickup_minutes",
    "is_empty_taxi"
).orderBy("medallion", "pickup_datetime").show(10, truncate=False)

Schema of DataFrame with empty taxi status:
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_time_in_secs: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_grid_coords: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- dropoff_grid_coords: array (nullable = true)
 |    |-- element: integer (conta

##Calculate number of empty taxis per dropoff grid area
To use the number of empty taxis in the profitability analysis, we need to group the empty taxis per drop-off area.

In [12]:
from pyspark.sql.functions import count, col

# Calculate the count of empty taxi instances per dropoff grid area
empty_taxi_count_per_dropoff_area = final_data_with_empty_status.filter(
    col("is_empty_taxi") == True
).groupBy("dropoff_grid_x", "dropoff_grid_y") \
 .agg(count("medallion").alias("empty_taxi_count"))

# Display the schema and first few rows of the resulting DataFrame
print("Schema of empty_taxi_count_per_dropoff_area:")
empty_taxi_count_per_dropoff_area.printSchema()
print("\nFirst 10 rows of empty_taxi_count_per_dropoff_area:")
empty_taxi_count_per_dropoff_area.show(10)

Schema of empty_taxi_count_per_dropoff_area:
root
 |-- dropoff_grid_x: integer (nullable = true)
 |-- dropoff_grid_y: integer (nullable = true)
 |-- empty_taxi_count: long (nullable = false)


First 10 rows of empty_taxi_count_per_dropoff_area:
+--------------+--------------+----------------+
|dropoff_grid_x|dropoff_grid_y|empty_taxi_count|
+--------------+--------------+----------------+
|           172|           151|            5279|
|           180|           154|             563|
|           196|           162|               9|
|           141|           173|              14|
|           161|           175|              59|
|           184|           158|              78|
|           188|           162|              32|
|           138|           172|              43|
|           177|           163|              93|
|           175|           178|              15|
+--------------+--------------+----------------+
only showing top 10 rows



##Calculate area profitability
Finally, calculate the area profitability by dividing the average area profit by the number of empty taxis in that area.